# CyberVision Gen — Modèle A : WGAN-GP conditionnel

Notebook Colab exécutable pour entraîner depuis zéro un WGAN-GP MLP sur les **63 caractéristiques** et **11 classes** de CIC-IDS2017. Le train entraîne les réseaux, la validation suit l'apprentissage et choisit les checkpoints. **Le test n'est jamais chargé ici.** Les modèles B et C sont hors périmètre.

> Avertissement : l'audit temporel actuel contient trois corrélations rang/temps inférieures à 0,9. Tant que ce point du pipeline n'est pas résolu ou validé, tout résultat expérimental reste provisoire.

Les encadrés « Exemple de sortie attendue » sont illustratifs, pas des résultats exécutés.

### Exemples de sorties attendues — purement illustratifs

| Section | Forme de sortie attendue (sans valeur expérimentale inventée) |
|---|---|
| A | version PyTorch et `device=cpu` ou `device=cuda` |
| B | trois chemins existants : dépôt, données privées, sorties Drive |
| C | formes train/validation, plage et dictionnaire des 11 effectifs |
| D | tenseur du générateur de forme `(4, 63)` dans `[-1,1]` |
| E | quatre scores réels de forme `(4,)` |
| F | une gradient penalty scalaire, finie et différentiable |
| G | une ligne JSON de métriques par époque et des fichiers `best.pt`/`latest.pt` |
| H | dictionnaire des 11 formes `(N,63)` et matrices `(N,1,8,8)` |
| I | figures de pertes, heatmaps et histogrammes train/validation |
| J | trois Parquet distincts, checkpoint, artefacts et `provenance.json` |

## A — Initialisation
Installation légère : Colab fournit normalement PyTorch avec CUDA. On n'écrase donc pas sa version GPU.

In [ ]:
%pip install -q pyarrow pandas pyyaml matplotlib seaborn pytest
import os, sys, json, random, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch={torch.__version__} | device={DEVICE}')

**Exemple de sortie attendue (illustratif)** : `PyTorch=2.x | device=cuda`.

## B — Google Drive et chemins persistants
Après une déconnexion, réexécuter A puis B. Le chemin n'est jamais supposé être `/content` : modifiez `PROJECT_ROOT` ou définissez `CYBERVISION_PROJECT_ROOT`.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Hors Colab : montage Drive ignoré.')

PROJECT_ROOT = Path(os.environ.get(
    'CYBERVISION_PROJECT_ROOT',
    '/content/drive/MyDrive/cybervision-gen'
)).expanduser()
if not PROJECT_ROOT.exists() and (Path.cwd() / 'cybervision_gan').exists():
    PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'cybervision_gan').exists():
    raise FileNotFoundError(
        f'Dépôt introuvable dans {PROJECT_ROOT}. Modifiez PROJECT_ROOT puis réexécutez cette cellule.'
    )
PROCESSED_DIR = Path(os.environ.get(
    'CYBERVISION_PROCESSED_DIR',
    str(PROJECT_ROOT / 'cybervision_data/data/processed')
))
ARTIFACTS_DIR = PROJECT_ROOT / 'cybervision_data/artifacts'
OUTPUT_DIR = Path(os.environ.get(
    'CYBERVISION_OUTPUT_DIR',
    str(PROJECT_ROOT / 'cybervision_gan/outputs/model_a')
))
required_data = [PROCESSED_DIR / 'train_scaled.parquet', PROCESSED_DIR / 'val_scaled.parquet']
missing = [str(path) for path in required_data if not path.is_file()]
if missing:
    raise FileNotFoundError('Parquet requis absent :\n- ' + '\n- '.join(missing) +
                            '\nCopiez-les sur Drive ou corrigez CYBERVISION_PROCESSED_DIR.')
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'cybervision_data'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Dépôt :', PROJECT_ROOT)
print('Données privées :', PROCESSED_DIR)
print('Sorties persistantes :', OUTPUT_DIR)

## C — Chargement et validation des données
L'ordre vient exclusivement de `matrix_layout.json`; les labels textuels viennent exclusivement de `label_encoding.json`. Le sampler inverse-fréquence équilibre les classes rares sans changer les partitions.

In [ ]:
from cybervision_gan.training.train import create_dataloader, set_seed
from cvgen_data.matrix import MatrixLayout

with open(PROJECT_ROOT / 'cybervision_gan/configs/model_a.yaml', encoding='utf-8') as handle:
    cfg = yaml.safe_load(handle)
with open(ARTIFACTS_DIR / 'matrix_layout.json', encoding='utf-8') as handle:
    layout_json = json.load(handle)
with open(ARTIFACTS_DIR / 'label_encoding.json', encoding='utf-8') as handle:
    label_to_index = json.load(handle)
features = layout_json['features']
assert len(features) == layout_json['n_features'] == cfg['model']['n_features'] == 63
assert len(label_to_index) == cfg['model']['n_classes'] == 11
assert sorted(label_to_index.values()) == list(range(11))
assert np.asarray(layout_json['mask']).sum() == 63
layout = MatrixLayout.from_json(str(ARTIFACTS_DIR / 'matrix_layout.json'))

train_df = pd.read_parquet(PROCESSED_DIR / 'train_scaled.parquet')
val_df = pd.read_parquet(PROCESSED_DIR / 'val_scaled.parquet')
for split_name, frame in [('train', train_df), ('validation', val_df)]:
    missing_columns = set(features + ['label']) - set(frame.columns)
    if missing_columns: raise KeyError(f'{split_name}: colonnes absentes {sorted(missing_columns)}')
    unknown = set(frame['label'].dropna().unique()) - set(label_to_index)
    if unknown: raise ValueError(f'{split_name}: labels inconnus {sorted(unknown)}')
    values = frame[features].to_numpy(dtype=np.float32)
    if not np.isfinite(values).all(): raise ValueError(f'{split_name}: NaN ou infini détecté')
    print(split_name, frame.shape, 'plage=', (float(values.min()), float(values.max())))

X_train = train_df[features].to_numpy(dtype=np.float32)
X_val = val_df[features].to_numpy(dtype=np.float32)
y_train = train_df['label'].map(label_to_index).to_numpy(dtype=np.int64)
y_val = val_df['label'].map(label_to_index).to_numpy(dtype=np.int64)
if X_train.min() < -1.0001 or X_train.max() > 1.0001:
    raise ValueError("Le train n'est pas dans la plage contractuelle [-1,1].")
if X_val.min() < -1.0001 or X_val.max() > 1.0001:
    print('Attention : validation hors plage train, autorisé par le pipeline (non-écrêté).')
set_seed(cfg['project']['seed'])
train_loader = create_dataloader(
    X_train, y_train, cfg['training']['batch_size'], balanced=True, seed=SEED,
    num_workers=cfg['training']['num_workers'], drop_last=True)
val_loader = create_dataloader(
    X_val, y_val, cfg['training']['batch_size'], balanced=True, seed=SEED + 1,
    num_workers=cfg['training']['num_workers'], drop_last=True)
print('Effectifs train :', train_df['label'].value_counts().to_dict())

**Exemple de sortie attendue (illustratif)** : deux formes `(N, 65 environ)`, plages proches de `[-1, 1]`, puis les 11 effectifs. Le nombre exact de colonnes inclut les métadonnées du pipeline.

## D — Générateur conditionnel
Le bruit et l'embedding de classe sont concaténés. La dernière activation `tanh` respecte `[-1,1]`.

In [ ]:
from cybervision_gan.models import Generator, Critic
def build_models(config):
    model = config['model']
    generator = Generator(model['latent_dim'], model['n_classes'], model['n_features'],
                          model['embedding_dim'], model['generator_hidden_dims'])
    critic = Critic(model['n_classes'], model['n_features'], model['embedding_dim'],
                    model['critic_hidden_dims'])
    return generator, critic
generator, critic = build_models(cfg)
z = torch.randn(4, cfg['model']['latent_dim'])
demo_labels = torch.tensor([label_to_index[x] for x in ['PortScan','Bot','DoS Slowhttptest','BENIGN']])
demo = generator(z, demo_labels)
print(generator)
print('Sortie G :', tuple(demo.shape), float(demo.min()), float(demo.max()))

## E — Critique conditionnel
Le critique reçoit le vecteur et le même label. Sa sortie est un score réel : aucune sigmoid ni BatchNorm.

In [ ]:
scores = critic(demo, demo_labels)
print(critic)
print('Scores C :', tuple(scores.shape), scores.detach().numpy())

## F — WGAN-GP et vérification des gradients
La pénalité interpole réel et synthétique **de même classe**. Le `detach` n'est utilisé que pour l'affichage; la mise à jour de G conserve son graphe.

In [ ]:
from cybervision_gan.training.gradient_penalty import gradient_penalty
real_batch, label_batch = next(iter(train_loader))
small_n = min(16, len(real_batch))
fake_batch = generator(torch.randn(small_n, cfg['model']['latent_dim']), label_batch[:small_n])
gp = gradient_penalty(critic, real_batch[:small_n], fake_batch, label_batch[:small_n])
gp.backward()
assert torch.isfinite(gp) and any(p.grad is not None for p in critic.parameters())
generator.zero_grad(set_to_none=True); critic.zero_grad(set_to_none=True)
print('Gradient penalty finie :', float(gp.detach()))

## G — Smoke test, entraînement et reprise
Le smoke test utilise des réseaux neufs et un dossier séparé. Il vérifie cinq batches (un cycle `n_critic`) sans contaminer le modèle complet. `best.pt` est sélectionné exclusivement avec la validation.

In [ ]:
from cybervision_gan.training.train import train_wgan_gp
smoke_g, smoke_c = build_models(cfg)
smoke_history = train_wgan_gp(
    smoke_g, smoke_c, train_loader, val_loader, cfg, OUTPUT_DIR / 'smoke_checkpoints',
    smoke_test=True)
assert len(smoke_history) == 1
print('Smoke test réussi. Ces métriques ne constituent pas un résultat expérimental.')

In [ ]:
# Passez à True uniquement après le smoke test. Les checkpoints restent sur Drive.
RUN_FULL_TRAINING = False
RESUME = True
generator, critic = build_models(cfg)
full_checkpoint_dir = OUTPUT_DIR / 'checkpoints'
latest = full_checkpoint_dir / 'latest.pt'
resume_from = latest if (RUN_FULL_TRAINING and RESUME and latest.exists()) else None
if RUN_FULL_TRAINING:
    history = train_wgan_gp(generator, critic, train_loader, val_loader, cfg,
                                full_checkpoint_dir, resume_from=resume_from)
else:
    history = smoke_history
    print('Entraînement complet désactivé. Fixez RUN_FULL_TRAINING=True pour le lancer/reprendre.')

**Exemple de sortie attendue (illustratif)** : une ligne JSON par époque avec `critic_loss`, `generator_loss`, `gradient_penalty` et `val_distribution_distance`. Aucune valeur numérique n'est anticipée ici.

## H — Génération conditionnelle et matrice 8×8
La cellule préfère le checkpoint complet; à défaut, elle utilise celui du smoke test uniquement pour vérifier le pipeline. Les 11 classes sont prises en charge, avec priorité à PortScan, Bot et DoS Slowhttptest.

In [ ]:
from cybervision_gan.evaluation.generate import generate_conditioned, vectors_to_matrices
from cybervision_gan.training.train import load_checkpoint
priority = cfg['project']['priority_classes']
best_full = OUTPUT_DIR / 'checkpoints/best.pt'
best_smoke = OUTPUT_DIR / 'smoke_checkpoints/best.pt'
selected_checkpoint = best_full if best_full.exists() else best_smoke
if not selected_checkpoint.exists(): raise FileNotFoundError('Aucun checkpoint : exécutez le smoke test.')
generator, critic = build_models(cfg)
state = load_checkpoint(selected_checkpoint, generator, critic, map_location=DEVICE)
if selected_checkpoint == best_smoke:
    print('ATTENTION : checkpoint smoke utilisé; sorties techniques, pas résultats scientifiques.')
samples_per_class = {name: (5000 if name in priority else 1000) for name in label_to_index}
generated_by_class = {
    name: generate_conditioned(generator, index, samples_per_class[name],
                               cfg['model']['latent_dim'], DEVICE, cfg['generation']['batch_size'])
    for name, index in label_to_index.items()
}
for name, values in generated_by_class.items():
    assert values.shape == (samples_per_class[name], 63)
priority_matrices = {name: vectors_to_matrices(generated_by_class[name], layout_json) for name in priority}
assert all(m.shape[1:] == (1, 8, 8) and np.all(m[:, 0, 7, 7] == 0)
           for m in priority_matrices.values())
print({name: values.shape for name, values in generated_by_class.items()})

## I — Visualisations pédagogiques
On affiche les pertes, une matrice réelle/synthétique, des histogrammes réel/synthétique, puis train/validation par classe. Ces figures ne consultent jamais le test.

In [ ]:
metrics = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, column in zip(axes, ['critic_loss','generator_loss','gradient_penalty']):
    ax.plot(metrics['epoch'], metrics[column]); ax.set_title(column); ax.set_xlabel('époque')
plt.tight_layout(); plt.show()

chosen_class = 'PortScan'
real_vector = train_df.loc[train_df.label == chosen_class, features].iloc[0].to_numpy(np.float32)[None]
real_matrix = layout.to_matrix(real_vector)[0, 0]
fake_matrix = priority_matrices[chosen_class][0, 0]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, matrix, title in zip(axes, [real_matrix, fake_matrix], ['Réel train','Synthétique brut']):
    sns.heatmap(matrix, vmin=-1, vmax=1, cmap='coolwarm', ax=ax); ax.set_title(f'{chosen_class} — {title}')
plt.tight_layout(); plt.show()

selected_features = ['flow_duration','flow_bytes_s','flow_packets_s']
fig, axes = plt.subplots(len(priority), len(selected_features), figsize=(15, 10))
for row, class_name in enumerate(priority):
    real = train_df[train_df.label == class_name]
    fake = generated_by_class[class_name]
    for col, feature in enumerate(selected_features):
        idx = features.index(feature); ax = axes[row, col]
        ax.hist(real[feature], bins=40, density=True, histtype='step', label='train réel')
        ax.hist(fake[:, idx], bins=40, density=True, histtype='step', label='synthétique')
        ax.set_title(f'{class_name} — {feature}'); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(len(priority), len(selected_features), figsize=(15, 10))
for row, class_name in enumerate(priority):
    tr = train_df[train_df.label == class_name]; va = val_df[val_df.label == class_name]
    for col, feature in enumerate(selected_features):
        ax = axes[row, col]
        ax.hist(tr[feature], bins=40, density=True, histtype='step', label='train')
        ax.hist(va[feature], bins=40, density=True, histtype='step', label='validation')
        ax.set_title(f'{class_name} — {feature}'); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

## J — Export traçable et transformation inverse
`synthetic_raw_inverse.parquet` est volontairement **non post-traité** afin que l'évaluation des violations physiques ne soit pas masquée. La version post-traitée est distincte. Ni Parquet ni checkpoint ne doivent être commités.

In [ ]:
from cvgen_data.scaling import FlowScaler
export_dir = OUTPUT_DIR / 'generated'
export_dir.mkdir(parents=True, exist_ok=True)
scaler = FlowScaler.from_json(str(ARTIFACTS_DIR / 'scaler.json'))
with open(ARTIFACTS_DIR / 'feature_schema.json', encoding='utf-8') as handle:
    feature_schema = json.load(handle)
normalized_parts = []
for class_name, values in generated_by_class.items():
    part = pd.DataFrame(values, columns=features)
    part['label'] = class_name
    part['label_index'] = label_to_index[class_name]
    normalized_parts.append(part)
normalized = pd.concat(normalized_parts, ignore_index=True)
raw_inverse = scaler.inverse_transform(normalized[features], postprocess=False)
raw_inverse['label'] = normalized['label']; raw_inverse['label_index'] = normalized['label_index']
postprocessed = scaler.inverse_transform(normalized[features], postprocess=True, schema=feature_schema)
postprocessed['label'] = normalized['label']; postprocessed['label_index'] = normalized['label_index']
normalized.to_parquet(export_dir / 'synthetic_normalized_unprocessed.parquet', index=False)
raw_inverse.to_parquet(export_dir / 'synthetic_raw_inverse.parquet', index=False)
postprocessed.to_parquet(export_dir / 'synthetic_postprocessed.parquet', index=False)
shutil.copy2(selected_checkpoint, export_dir / 'selected_checkpoint.pt')
shutil.copy2(PROJECT_ROOT / 'cybervision_gan/configs/model_a.yaml', export_dir / 'model_a.yaml')
for name in ['label_encoding.json','matrix_layout.json','scaler.json','feature_schema.json']:
    shutil.copy2(ARTIFACTS_DIR / name, export_dir / name)
provenance = {
    'seed': SEED, 'checkpoint': str(selected_checkpoint), 'features': features,
    'label_encoding': label_to_index, 'uses_train': True, 'uses_validation': True,
    'uses_test': False, 'results_provisional_timestamp_audit': True,
}
(export_dir / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
print('Exports :', sorted(path.name for path in export_dir.iterdir()))

### Rapport d'exécution à compléter après le run

- Tests réellement exécutés dans ce notebook : contrôles de schéma/plage, gradient penalty, smoke test et padding `(7,7)=0`.
- Non vérifiable sans les Parquet privés et un run GPU complet : convergence, temps d'entraînement, qualité statistique finale et violations physiques observées.
- Ne pas reporter les sorties du smoke test comme résultats.
- Le test demeure réservé au protocole d'évaluation séparé.